In [1]:
## Import des bibliiothèques
import pandas as pd
import numpy as np

In [2]:
# === FICHIERS ===
input_file = "C:/Users/Lenovo i7/OneDrive - ENSEA/Stage CGF Gestion/Documents/Mémoire/Data/EDOUARD/Données Modele HMM-FSHMM(4).xlsx"
output_file = "C:/Users/Lenovo i7/OneDrive - ENSEA/Stage CGF Gestion/Documents/Mémoire/Data/EDOUARD/Rendement/rendements_portefeuilles_long.xlsx"


In [3]:
# === CHARGEMENT DES RENDEMENTS JOURNALIERS DES ACTIFS ===
rendements = pd.read_excel(input_file, sheet_name="Momentum(i) Rendement Journalie")
rendements.rename(columns={rendements.columns[0]: "Date"}, inplace=True)
rendements["Date"] = pd.to_datetime(rendements["Date"])
rendements.set_index("Date", inplace=True)
rendements.sort_index(inplace=True)

In [4]:
# === CHARGER TOUTES LES FEUILLES COMME FACTEURS (INCLUS momentum) ===
xls = pd.ExcelFile(input_file)
facteurs = xls.sheet_names  # Inclut "Momentum(i) Rendement Journalie"


In [5]:
# === STOCKAGE DES RÉSULTATS ===
portefeuille_rendements = {}

for facteur in facteurs:
    print(f"Traitement du facteur : {facteur}")
    
    df = pd.read_excel(input_file, sheet_name=facteur)
    df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    df["Date"] = pd.to_datetime(df["Date"])
    df.set_index("Date", inplace=True)
    df.sort_index(inplace=True)

    rendement_portefeuille = pd.Series(index=rendements.index, dtype="float64")

    monthly_groups = df.groupby([df.index.year, df.index.month])

    for (year, month), group in monthly_groups:
        try:
            start_date = group.index.min()
            end_date = group.index.max()

            # Nettoyage et conversion en numérique
            factor_scores = df.loc[start_date].apply(pd.to_numeric, errors='coerce').dropna()
            factor_scores = factor_scores[factor_scores.index.isin(rendements.columns)]

            # Trier les actifs selon le facteur
            facteur_minimisation = ["volatilité(i) la volatilité".lower(), "risk(i) beta".lower()]
            if facteur.strip().lower() in facteur_minimisation:
                sorted_assets = factor_scores.sort_values(ascending=True)  # On MINIMISE
            else:
                sorted_assets = factor_scores.sort_values(ascending=False)  # On MAXIMISE
                
            n = len(sorted_assets)
            top_20 = sorted_assets.head(int(0.2 * n)).index
            bottom_20 = sorted_assets.tail(int(0.2 * n)).index

            # Poids des actifs
            weights = pd.Series(0, index=rendements.columns, dtype="float64")
            weights[top_20] = 1 / len(top_20)
            #weights[bottom_20] = -1 / len(bottom_20)

            # Rendements journaliers du portefeuille
            monthly_returns = rendements.loc[start_date:end_date].dot(weights)
            rendement_portefeuille.loc[start_date:end_date] = monthly_returns

        except Exception as e:
            print(f"⚠️ Erreur en {year}-{month} pour le facteur {facteur} : {e}")

    portefeuille_rendements[facteur] = rendement_portefeuille


Traitement du facteur : Value (i)Booktomarket global
Traitement du facteur : Value(ii) ResultNet-Cours
Traitement du facteur : Value(iv) CA-cours
Traitement du facteur : Value(v) EBIT-EVapproximé
Traitement du facteur : Quality(i) Levier Financier
Traitement du facteur : Quality(ii) ROE
Traitement du facteur : Qaulity(iii) ROA
Traitement du facteur : Quality(iv) Altman Z-Score
Traitement du facteur : Growth(i) Divident yield
Traitement du facteur : Volatilité(i) la volatilité
Traitement du facteur : Momentum(i) Rendement Journalie
Traitement du facteur : Mom(ii) 6 Month Price Momentum
Traitement du facteur : Mom(iii) 12 Month Price momentu
Traitement du facteur : Liquidité(i) volume
Traitement du facteur : Risk(i) Beta
Traitement du facteur : Size(i) Capitalisation boursièr


In [6]:
# === RÉUNION DES RÉSULTATS DANS UN TABLEAU FINAL ===
resultats = pd.DataFrame(portefeuille_rendements)


In [ ]:
resultats

In [9]:
# === EXPORT VERS EXCEL ===
resultats.to_excel(output_file)
print(f"\n✅ Fichier de résultats généré avec succès : {output_file}")



✅ Fichier de résultats généré avec succès : C:/Users/Lenovo i7/OneDrive - ENSEA/Stage CGF Gestion/Documents/Mémoire/Data/EDOUARD/Rendement/rendements_portefeuilles_long.xlsx
